## CPSC 4970 - Jonathan Braun - M6 Notebook 2: SOM + MLP

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
penguins = pd.read_csv("penguins.csv", na_values=["?", "_"])

print("Original shape:", penguins.shape)
print("\nMissing values before dropping:")
print(penguins.isna().sum())

penguins = penguins.dropna().copy()

print("\nCleaned shape:", penguins.shape)
print("\nMissing values after dropping:")
print(penguins.isna().sum())

penguins.head()

In [ ]:
numeric_features = [
    "culmen_length_mm",
    "culmen_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

categorical_features = [
    "island",
    "sex"
]

X_numeric = penguins[numeric_features]
X_categorical = penguins[categorical_features]
y = penguins["species"]

print("Numerical features:")
display(X_numeric.head())

print("Categorical features:")
display(X_categorical.head())

print("Target counts:")
print(y.value_counts())

## Train/test split

In [ ]:
X_train_numeric, X_test_numeric, X_train_categorical, X_test_categorical, y_train, y_test = train_test_split(
    X_numeric,
    X_categorical,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training numerical shape:", X_train_numeric.shape)
print("Testing numerical shape:", X_test_numeric.shape)
print("Training categorical shape:", X_train_categorical.shape)
print("Testing categorical shape:", X_test_categorical.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_numeric)
X_test_scaled = scaler.transform(X_test_numeric)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)

## SOM implementation

In [ ]:
class SimpleSOM:
    def __init__(self, m, n, dim, learning_rate=0.5, sigma=None, n_iterations=3000, random_state=None):
        self.m = m
        self.n = n
        self.dim = dim
        self.learning_rate = learning_rate
        self.sigma = sigma if sigma is not None else max(m, n) / 2
        self.n_iterations = n_iterations
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)
        self.locations = np.array([(i, j) for i in range(m) for j in range(n)])
        self.weights = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)

        # Initialize SOM weights using random samples from the training data
        initial_indices = self.rng.choice(len(X), size=self.m * self.n, replace=True)
        self.weights = X[initial_indices].copy()

        for t in range(self.n_iterations):
            x = X[self.rng.integers(len(X))]
            bmu_index = self.winner_index(x)

            progress = t / max(1, self.n_iterations - 1)
            current_learning_rate = self.learning_rate * np.exp(-progress)
            current_sigma = max(self.sigma * np.exp(-progress), 1e-6)

            distances_squared = np.sum(
                (self.locations - self.locations[bmu_index]) ** 2,
                axis=1
            )

            neighborhood = np.exp(
                -distances_squared / (2 * current_sigma * current_sigma)
            )[:, np.newaxis]

            self.weights += current_learning_rate * neighborhood * (x - self.weights)

        return self

    def winner_index(self, x):
        distances = np.linalg.norm(self.weights - x, axis=1)
        return int(np.argmin(distances))

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self.winner_index(x) for x in X])

    def labels_as_coordinates(self, X):
        winner_indices = self.predict(X)
        coordinates = self.locations[winner_indices]
        return np.array([f"{row}_{col}" for row, col in coordinates])

## Train the SOM and create categories

In [ ]:
som = SimpleSOM(
    m=3,
    n=3,
    dim=4,
    learning_rate=0.5,
    sigma=1.5,
    n_iterations=3000,
    random_state=7
)

som.fit(X_train_scaled)

train_som_categories = som.labels_as_coordinates(X_train_scaled)
test_som_categories = som.labels_as_coordinates(X_test_scaled)

print("Training SOM categories:")
print(pd.Series(train_som_categories).value_counts().sort_index())

print("\nTesting SOM categories:")
print(pd.Series(test_som_categories).value_counts().sort_index())